<a href="https://colab.research.google.com/github/kumakun2008/CIFAR-10_Comp/blob/main/kumakun2008_ConvNeXt_1116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 画像分類（ImageNet事前学習済みモデル ファインチューニング：ConvNeXt-Small + Swin-S + ViT-B/16）

## 工夫した点（箇条書き）
* **ImageNetで事前学習したモデルの重みを使用した**（`torchvision.models` の `convnext_small`, `swin_s`, `vit_b_16` の `weights='DEFAULT'` を利用し、出力層を10クラスに差し替えて本ノートブック内でCIFAR-10全層ファインチューニングを実施）。
* **系統の異なる3アーキテクチャの選定**：
  * モダンCNNの **ConvNeXt-Small**、階層型Transformerの **Swin-S**、グローバルAttention型Transformerの **ViT-B/16** という特徴抽出の仕組みが異なる3モデルを採用した。
* **前処理・ルール準拠の正規化とデータ拡張**：
  * 前処理の正規化統計（`means`, `stds`）は配布コードのとおり **CIFAR-10訓練データから計算した値** をそのまま使用した。
  * 事前学習モデルの入力解像度に合わせて `Resize((224, 224), BICUBIC)` を行い、訓練時は `RandomHorizontalFlip(0.5)`・`TrivialAugmentWide`・`RandomErasing`・`Mixup(alpha=0.2)` を組み合わせて過学習を抑制した。
* **ファインチューニング向けの最適化設定**：
  * 配布コードのループ内 `optimizer` 再初期化を修正し、**AdamW** ＋ **Linear Warmup付きCosineAnnealing** ＋ **勾配クリッピング（max_norm=1.0）** ＋ **Label Smoothing（0.05）** を導入した。
  * `bfloat16` / `float16` の自動混合精度（AMP）で高速かつ安定した学習を行った。
* **検証（Validation）に基づく確認と仕上げ（テストデータは学習・選定に不使用）**：
  * まず訓練45,000枚 / 検証5,000枚に分割し、`Val. Acc`・検証データの混同行列・誤分類可視化で学習の妥当性を確認した。
  * 設定確定後、訓練＋検証を合わせた **50,000枚全量** で3モデルをそれぞれファインチューニングした。
  * 最終テスト評価では、各モデル単体の `Test Acc` に加え、**TTA（元画像＋左右反転画像の確率平均）と3モデルのアンサンブル**を適用した。


In [ ]:
import os
import math
import copy
import time
import random
import shutil
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision import models

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
SEED = 1234

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# CIFAR-10 データの読み込みと前処理

In [ ]:
ROOT = '.data'
os.makedirs(ROOT, exist_ok=True)
tar_path = os.path.join(ROOT, 'cifar-10-python.tar.gz')
drive_tar_path = '/content/drive/MyDrive/Colab_Data/CIFAR10/cifar-10-python.tar.gz'
mirror_url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'

if not os.path.exists(os.path.join(ROOT, 'cifar-10-batches-py')) and (
    not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071
):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        if os.path.exists(drive_tar_path):
            print(f'Copying CIFAR-10 from Google Drive: {drive_tar_path}')
            shutil.copy2(drive_tar_path, tar_path)
    except Exception as e:
        print(f'Google Drive copy skipped ({e})')

    if not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071:
        print(f'Downloading CIFAR-10 from mirror: {mirror_url}')
        urllib.request.urlretrieve(mirror_url, tar_path)

train_data = datasets.CIFAR10(root=ROOT,
                              train=True,
                              download=True)

means = train_data.data.mean(axis=(0, 1, 2)) / 255
stds = train_data.data.std(axis=(0, 1, 2)) / 255

print(f'Calculated means: {means}')
print(f'Calculated stds: {stds}')


In [ ]:
IMAGE_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=means, std=stds),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=means, std=stds),
])


In [ ]:
full_train_data = datasets.CIFAR10(ROOT,
                                   train=True,
                                   download=True,
                                   transform=train_transforms)

test_data = datasets.CIFAR10(ROOT,
                             train=False,
                             download=True,
                             transform=test_transforms)

VALID_RATIO = 0.9

n_train_examples = int(len(full_train_data) * VALID_RATIO)
n_valid_examples = len(full_train_data) - n_train_examples

split_generator = torch.Generator().manual_seed(SEED)
train_data, valid_data = data.random_split(full_train_data,
                                           [n_train_examples, n_valid_examples],
                                           generator=split_generator)

valid_data = copy.deepcopy(valid_data)
valid_data.dataset.transform = test_transforms

print(f'Number of training examples (Phase 1): {len(train_data)}')
print(f'Number of validation examples (Phase 1): {len(valid_data)}')
print(f'Number of full training examples (Phase 2): {len(full_train_data)}')
print(f'Number of testing examples: {len(test_data)}')


In [ ]:
def normalize_image(image):
    image = image.clone()
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image

def plot_images(images, labels, classes, normalize=False):
    n_images = len(images)
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(10, 10))
    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i + 1)
        image = images[i]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')


In [ ]:
N_IMAGES = 25

images, labels = zip(*[(image, label) for image, label in
                       [train_data[i] for i in range(N_IMAGES)]])

classes = test_data.classes
plot_images(images, labels, classes, normalize=True)


# DataLoader と GPU/AMP 設定

In [ ]:
BATCH_SIZE = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = (use_amp and amp_dtype == torch.float16)
num_workers = min(4, os.cpu_count() or 2) if device.type == 'cuda' else 0
pin_memory = (device.type == 'cuda')

train_iterator = data.DataLoader(train_data,
                                 shuffle=True,
                                 batch_size=BATCH_SIZE,
                                 num_workers=num_workers,
                                 pin_memory=pin_memory)

valid_iterator = data.DataLoader(valid_data,
                                 batch_size=BATCH_SIZE,
                                 num_workers=num_workers,
                                 pin_memory=pin_memory)

full_train_iterator = data.DataLoader(full_train_data,
                                      shuffle=True,
                                      batch_size=BATCH_SIZE,
                                      num_workers=num_workers,
                                      pin_memory=pin_memory)

test_iterator = data.DataLoader(test_data,
                                batch_size=BATCH_SIZE,
                                num_workers=num_workers,
                                pin_memory=pin_memory)

print(f'Device: {device} | AMP: {use_amp} ({amp_dtype}) | num_workers: {num_workers}')


# モデル設計（ImageNet事前学習済みモデル × 3系統）
1. **ConvNeXt-Small** (`convnext_small`, `weights='DEFAULT'`)
2. **Swin-Small** (`swin_s`, `weights='DEFAULT'`)
3. **ViT-B/16** (`vit_b_16`, `weights='DEFAULT'`)


In [ ]:
OUTPUT_DIM = 10

def build_convnext_small(num_classes=10):
    model = models.convnext_small(weights='DEFAULT')
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

def build_swin_small(num_classes=10):
    model = models.swin_s(weights='DEFAULT')
    in_features = model.head.in_features
    model.head = nn.Linear(in_features, num_classes)
    return model

def build_vit_b16(num_classes=10):
    model = models.vit_b_16(weights='DEFAULT')
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    return model

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Model builders ready.')


# 学習・評価関数（AdamW / Warmup+Cosine / Mixup / AMP / TTA対応）

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc


def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


def create_warmup_cosine_scheduler(optimizer, epochs, warmup_epochs=2, min_lr_ratio=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train(model, iterator, optimizer, criterion, device, scaler, mixup_alpha=0.2,
          use_amp=True, amp_dtype=torch.float16, max_grad_norm=1.0):
    epoch_loss = 0.0
    epoch_acc = 0.0

    model.train()

    for (x, y) in tqdm(iterator, desc="Training", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if mixup_alpha > 0:
            x_mixed, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x_mixed)
                loss = mixup_criterion(criterion, y_pred, y_a, y_b, lam)
            acc = lam * calculate_accuracy(y_pred, y_a) + (1.0 - lam) * calculate_accuracy(y_pred, y_b)
        else:
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


def evaluate(model, iterator, criterion, device, use_amp=True, amp_dtype=torch.float16):
    epoch_loss = 0.0
    epoch_acc = 0.0

    model.eval()

    with torch.no_grad():
        for (x, y) in tqdm(iterator, desc="Evaluating", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)

            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


# Phase 1：検証データ（Val. Acc）を用いたファインチューニング設定の確認
まずは `ConvNeXt-Small` を使って、訓練45,000枚 / 検証5,000枚で学習曲線と `Val. Acc` を確認します（テストデータは一切参照しません）。


In [ ]:
EPOCHS = 15
WEIGHT_DECAY = 1e-2
MIXUP_ALPHA = 0.2
LABEL_SMOOTHING = 0.05

set_seed(SEED)
val_model = build_convnext_small(OUTPUT_DIM).to(device)
print(f'ConvNeXt-Small trainable parameters: {count_parameters(val_model):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
eval_criterion = nn.CrossEntropyLoss().to(device)

optimizer = optim.AdamW(val_model.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
scheduler = create_warmup_cosine_scheduler(optimizer, epochs=EPOCHS, warmup_epochs=2)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

best_valid_acc = 0.0
best_epoch = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

for epoch in trange(EPOCHS, desc="Phase 1 (Val Check)"):
    start_time = time.monotonic()

    train_loss, train_acc = train(
        val_model, train_iterator, optimizer, criterion, device, scaler,
        mixup_alpha=MIXUP_ALPHA, use_amp=use_amp, amp_dtype=amp_dtype
    )
    valid_loss, valid_acc = evaluate(
        val_model, valid_iterator, eval_criterion, device, use_amp=use_amp, amp_dtype=amp_dtype
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(val_model.state_dict(), 'convnext_val_best.pt')

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s | LR: {scheduler.get_last_lr()[0]:.6f}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

print(f'\nBest Val. Acc: {best_valid_acc*100:.2f}% (at Epoch {best_epoch})')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['valid_loss'], label='Valid Loss')
axes[0].set_title('Loss Curve (Phase 1)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot([a * 100 for a in history['train_acc']], label='Train Acc')
axes[1].plot([a * 100 for a in history['valid_acc']], label='Valid Acc')
axes[1].set_title('Accuracy Curve (Phase 1)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
plt.show()


## 検証データ（Validation）での誤差分析（混同行列・誤分類画像）
テストデータを分析に使うのではなく、検証データで弱点を確認します。


In [ ]:
def get_predictions(model, iterator, device, use_tta=False, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    images, labels, probs = [], [], []
    with torch.no_grad():
        for (x, y) in iterator:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                y_prob = F.softmax(y_pred.float(), dim=-1)
                if use_tta:
                    y_pred_flip = model(torch.flip(x, dims=[3]))
                    y_prob = 0.5 * (y_prob + F.softmax(y_pred_flip.float(), dim=-1))
            images.append(F.interpolate(x.cpu(), size=(64, 64), mode='bilinear', align_corners=False))
            labels.append(y.cpu())
            probs.append(y_prob.cpu())
    return torch.cat(images, dim=0), torch.cat(labels, dim=0), torch.cat(probs, dim=0)


def plot_confusion_matrix(labels, pred_labels, classes):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = confusion_matrix(labels, pred_labels)
    cm = ConfusionMatrixDisplay(cm, display_labels=classes)
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    plt.xticks(rotation=20)


def plot_most_incorrect(incorrect, classes, n_images, normalize=True):
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(20, 16))
    for i in range(min(rows * cols, len(incorrect))):
        ax = fig.add_subplot(rows, cols, i + 1)
        image, true_label, probs = incorrect[i]
        image = image.permute(1, 2, 0)
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        true_class = classes[true_label]
        incorrect_class = classes[incorrect_label]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.cpu().numpy())
        ax.set_title(f'true: {true_class} ({true_prob:.3f})\npred: {incorrect_class} ({incorrect_prob:.3f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.4)


val_model.load_state_dict(torch.load('convnext_val_best.pt', map_location=device))
val_images, val_labels, val_probs = get_predictions(
    val_model, valid_iterator, device, use_tta=True, use_amp=use_amp, amp_dtype=amp_dtype
)
val_preds = torch.argmax(val_probs, dim=1)
print(f'Validation Acc (with TTA): {(val_preds == val_labels).float().mean().item()*100:.2f}%')
plot_confusion_matrix(val_labels, val_preds, classes)


In [ ]:
corrects = torch.eq(val_labels, val_preds)
incorrect_examples = []
for image, label, prob, correct in zip(val_images, val_labels, val_probs, corrects):
    if not correct:
        incorrect_examples.append((image, label, prob))

incorrect_examples.sort(reverse=True, key=lambda x: torch.max(x[2], dim=0).values)
plot_most_incorrect(incorrect_examples, classes, 25)


# Phase 2：訓練＋検証の全50,000枚で3モデルをファインチューニング（アンサンブル用）
Phase 1 の設定をもとに、訓練＋検証の **50,000枚すべて** で系統の異なる3つのImageNet事前学習モデルをそれぞれ学習します（`val_model` は不要になるため先にメモリを解放します）。
1. **ConvNeXt-Small**（純粋CNN系）
2. **Swin-Small**（階層型Transformer系）
3. **ViT-B/16**（グローバルAttention Transformer系）


In [ ]:
del val_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

def train_full_dataset(model_fn, model_name, save_path, lr=1e-4, epochs=15, seed=1234):
    set_seed(seed)
    model = model_fn().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = create_warmup_cosine_scheduler(optimizer, epochs=epochs, warmup_epochs=2)
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

    print(f'=== Start Full Fine-tuning: {model_name} ({count_parameters(model):,} params, lr={lr}) ===')
    for epoch in trange(epochs, desc=model_name):
        start_time = time.monotonic()
        train_loss, train_acc = train(
            model, full_train_iterator, optimizer, criterion, device, scaler,
            mixup_alpha=MIXUP_ALPHA, use_amp=use_amp, amp_dtype=amp_dtype
        )
        scheduler.step()
        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        if (epoch + 1) % 5 == 0 or epoch == 0 or (epoch + 1) == epochs:
            print(f'[{model_name}] Epoch {epoch+1:02}/{epochs} ({epoch_mins}m {epoch_secs}s) | '
                  f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')

    torch.save(model.state_dict(), save_path)
    print(f'Saved {model_name} weights to {save_path}\n')
    return model


In [ ]:
FULL_EPOCHS = 15

model_convnext = train_full_dataset(
    lambda: build_convnext_small(OUTPUT_DIM),
    model_name='ConvNeXt-Small',
    save_path='convnext_small_full.pt',
    lr=1e-4,
    epochs=FULL_EPOCHS,
    seed=SEED
)

model_swin = train_full_dataset(
    lambda: build_swin_small(OUTPUT_DIM),
    model_name='Swin-Small',
    save_path='swin_small_full.pt',
    lr=1e-4,
    epochs=FULL_EPOCHS,
    seed=SEED + 1
)

model_vit = train_full_dataset(
    lambda: build_vit_b16(OUTPUT_DIM),
    model_name='ViT-B/16',
    save_path='vit_b16_full.pt',
    lr=5e-5,
    epochs=FULL_EPOCHS,
    seed=SEED + 2
)


# 最終テスト評価（10,000枚）

In [ ]:
# 各モデル単体のテスト精度（配布コード準拠の評価）
model_convnext.load_state_dict(torch.load('convnext_small_full.pt', map_location=device))
model_swin.load_state_dict(torch.load('swin_small_full.pt', map_location=device))
model_vit.load_state_dict(torch.load('vit_b16_full.pt', map_location=device))

for name, m in [('ConvNeXt-Small', model_convnext),
                ('Swin-Small', model_swin),
                ('ViT-B/16', model_vit)]:
    test_loss, test_acc = evaluate(
        m, test_iterator, eval_criterion, device, use_amp=use_amp, amp_dtype=amp_dtype
    )
    print(f'[{name}] Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


In [ ]:
# 3モデル アンサンブル + TTA（左右反転平均）による最終テスト評価
def evaluate_ensemble_tta(models_list, iterator, device, use_tta=True, use_amp=True, amp_dtype=torch.float16):
    for m in models_list:
        m.eval()

    epoch_loss = 0.0
    epoch_acc = 0.0

    with torch.no_grad():
        for (x, y) in tqdm(iterator, desc="Testing Ensemble + TTA", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            probs_sum = 0.0
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                for m in models_list:
                    probs = F.softmax(m(x).float(), dim=-1)
                    if use_tta:
                        probs_flip = F.softmax(m(torch.flip(x, dims=[3])).float(), dim=-1)
                        probs = 0.5 * (probs + probs_flip)
                    probs_sum = probs_sum + probs

            avg_probs = probs_sum / len(models_list)
            loss = F.nll_loss(torch.log(avg_probs.clamp(min=1e-12)), y)
            acc = calculate_accuracy(avg_probs, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


ensemble_models = [model_convnext, model_swin, model_vit]
test_loss, test_acc = evaluate_ensemble_tta(
    ensemble_models, test_iterator, device, use_tta=True, use_amp=use_amp, amp_dtype=amp_dtype
)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')
